In [ ]:
import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook

In [ ]:
torch_precision = torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.05
n_iter = 450

In [ ]:
global_seed = 3 # Can be None

In [ ]:
# Cross-fade Composite Loss: CLIP <-> Normalizing Flow
from utils.losses.composite import CompositeLoss
from utils.losses.normalizing_flow import create_normalizing_flow_loss
from utils.losses.clip import CLIPDirectionalCosineSimilarity

def _clip_weight(step, total):
    t = float(step) / float(max(1, total))
    return max(0.0, 1.0 - t)

def _flow_weight(step, total):
    return 0.01 * (1.0 - _clip_weight(step, total)) # The flow weight is a lot smaller because negative log likelihood will dominate here. This is a little bigt of an interesting problem, because the best that CLIPDirectionalCosineSimilarity can get is 0, whereas the NLL of the flow model can be arbitrarily negative.


def make_composite_clip_normalizing_flow_crossfade_loss(clip_loss, flow_loss):
    """Return a CompositeLoss that crossfades from `clip_loss` to a normalizing flow loss.

    Args:
        clip_loss: an instance of a loss (e.g., `CLIPCosineSimilarity`).
        flow_loss: an instance of a loss (e.g., `NormalizingFlowLoss`).
    """

    return CompositeLoss([
        (_clip_weight, clip_loss),
        (_flow_weight, flow_loss),
    ])


In [ ]:
import open_clip
clip_model_name = 'ViT-B-16-SigLIP-512'
clip_pretrained = 'webli'
model, _, preprocess_eval = open_clip.create_model_and_transforms(clip_model_name, pretrained=clip_pretrained, device=device)
tokenizer = open_clip.get_tokenizer(clip_model_name)

In [ ]:
from scenes import SpringScene, SciFiRobotScene, CarScene, BlenderManScene, HouseScene, DinoScene, FlowerPotScene, RedCarScene, CandleScene, CarStudioScene
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())
scene = SciFiRobotScene(device=device)

In [ ]:
from utils.losses.clip import CLIPDirectionalCosineSimilarity

initial_prompt = 'ugly, uninteresting lighting'
target_prompt = 'beautiful night time lighting'

clip_loss = CLIPDirectionalCosineSimilarity(initial_prompt, target_prompt, scene.get_combined_image(color_space_converter).permute(2, 1, 0), model, tokenizer, device=device, preprocess=preprocess_eval, always_prenormalize_vectors=True)

In [ ]:
from utils.train import train_with_criterion


flow_loss = create_normalizing_flow_loss()
# criterion = make_composite_clip_normalizing_flow_crossfade_loss(clip_loss=clip_loss, flow_loss=flow_loss)
criterion = flow_loss # Ablation: normalizing flow only

output_directory = 'image_adjustment_tests'

scenes = [SpringScene, SciFiRobotScene, CarScene, BlenderManScene, HouseScene, DinoScene, FlowerPotScene, RedCarScene, CandleScene, CarStudioScene]
for scene in scenes:

    train_with_criterion(
        scene,
        lr, n_iter, criterion,
        starting_multiplier_std=(0.4, 0.4, 0.4),
        output_subdirectory_name=output_directory,
        n_results=4,
        torch_precision=torch_precision,
        render_color_space_converter=color_space_converter,
        require_physically_plausible_multipliers=True,
        title_prefix="CLIP Mixed with Normalizing Flow Loss Test no augmentation",
        device=device,
        save_every=50,
        model_name='normalizing_flow',
        pretrained_source='',
        seed=global_seed
    )

## Result Collection Helper
Run the following cell to collect all loss plots and final image grids from a specific output directory into a single summary folder. This makes it easier to compare runs side-by-side.

In [ ]:
# from utils.experiment import FolderManager

# # Initialize FolderManager with the subdirectory and call collect_results
# folder_manager = FolderManager(output_directory)
# folder_manager.collect_results()